#Imports

In [ ]:
import json
import os
import platform
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.datasets import mnist
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

# Render saved figures inline when executed in a notebook kernel.
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('matplotlib', 'inline')

In [ ]:
# Allocate GPU memory as needed rather than reserving the entire device.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

if os.environ.get('MNIST_REQUIRE_GPU') == '1' and not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Local GPU training requested, but TensorFlow cannot see a GPU.')

SEED = 42
keras.utils.set_random_seed(SEED)  # Python, NumPy, and TensorFlow seeds
tf.config.experimental.enable_op_determinism()

# Works when launched from the repository root or its notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL_DIR = PROJECT_ROOT / 'models'
CHECKPOINT_DIR = MODEL_DIR / 'mlp_checkpoints'
RESULT_DIR = PROJECT_ROOT / 'results' / 'mlp'
for folder in (MODEL_DIR, CHECKPOINT_DIR, RESULT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
environment = {
    'python': sys.version.split()[0],
    'tensorflow': tf.__version__,
    'keras': keras.__version__,
    'scikit-learn': sklearn.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': plt.matplotlib.__version__,
    'platform': platform.platform(),
    'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
    'gpu_details': [tf.config.experimental.get_device_details(d) for d in tf.config.list_physical_devices('GPU')],
    'logical_cpus': os.cpu_count(),
    'processor': platform.processor(),
}
print(json.dumps(environment, indent=2))
print('Project directory:', PROJECT_ROOT)

#Data

In [ ]:
(X_train_full, y_train_full), (X_test_raw, y_test) = mnist.load_data()

In [ ]:
print(X_train_full.shape, y_train_full.shape)
print(X_test_raw.shape, y_test.shape)

Optional one-hot encoding for the labels.

Instead of representing a label as a single integer, one-hot encoding converts it into a vector of length 10.

Structure: This vector contains all zeros, except for a "1" at the index corresponding to the actual digit.
Purpose: With `categorical_crossentropy`, this vector is the target for the ten class probabilities. Integer labels with `sparse_categorical_crossentropy` are equally valid classification targets; they do not turn the task into regression.

In [ ]:
# Example only: training below keeps integer labels with the sparse loss.
one_hot_examples = keras.utils.to_categorical(y_train_full[:10], 10)

One-hot encoding is like a multiple-choice answer key.

We give each digit 10 separate boxes (labeled 0 to 9).
For a label, we put a 1 in the correct box and a 0 in all the others.
This is an alternative to the integer labels used by this notebook.

In [ ]:
for i in range(10):
    print(one_hot_examples[i])

#Prepare data

In [ ]:
# Split the 60,000 training images into 50,000 train + 10,000 validation.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_full, y_train_full,  # the data to split (images and their labels together)
    test_size=10000,            # how many go into the second part (validation)
    random_state=SEED,         # fixed seed -> same shuffle every time, for everyone
    stratify=y_train_full,     # keep the same share of each digit in both parts
)

In [ ]:
#normalize data
X_train = X_train_raw.astype('float32') / 255.0
X_val = X_val_raw.astype('float32') / 255.0
X_test = X_test_raw.astype('float32') / 255.0

In [ ]:
#Reshaping the data into 1D vector to pass it (flattening)
X_train = X_train.reshape(X_train.shape[0], -1)
X_val = X_val.reshape(X_val.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [ ]:
assert X_train.shape == (50000, 784)
assert X_val.shape == (10000, 784)
assert X_test.shape == (10000, 784)
assert X_train.dtype == np.float32
assert 0.0 <= X_train.min() <= X_train.max() <= 1.0
assert set(np.unique(y_train)) == set(range(10))
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
print('Training label counts:', np.bincount(y_train))

# The actual mlp

In [ ]:
MAX_EPOCHS = 20
PATIENCE = 3
EXPERIMENTS = [
    {'name': 'A', 'hidden_units': [128, 128], 'learning_rate': 0.001, 'batch_size': 128, 'l2': 0.0},
    {'name': 'B', 'hidden_units': [256, 128], 'learning_rate': 0.001, 'batch_size': 128, 'l2': 0.0},
    {'name': 'C', 'hidden_units': [128, 128], 'learning_rate': 0.0003, 'batch_size': 128, 'l2': 0.0},
    {'name': 'D', 'hidden_units': [128, 128], 'learning_rate': 0.001, 'batch_size': 512, 'l2': 0.0},
    {'name': 'E', 'hidden_units': [128, 128], 'learning_rate': 0.001, 'batch_size': 128, 'l2': 0.0001},
]

In [ ]:
def build_mlp(hidden_units, learning_rate, l2):
    model = keras.Sequential(name='mnist_mlp')  # Sequential() adds layers in order
    model.add(keras.Input(shape=(784,)))
    penalty = regularizers.l2(l2) if l2 > 0 else None
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu', kernel_regularizer=penalty))
    model.add(layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

preview = build_mlp([128, 128], learning_rate=0.001, l2=0.0)
preview.summary()
del preview

This code builds a neural network that takes **784 input values** and predicts **one of 10 classes**—for example, recognizing digits from 0 to 9.

```python
model = Sequential()
```

Creates an empty model where layers run **in order**. Each layer’s output becomes the next layer’s input.

### 1. First hidden layer

```python
model.add(Dense(128, input_shape=(784,), activation='relu'))
```

- Receives **784 values** per sample, such as a flattened 28 × 28 image.
- Contains **128 neurons**, each connected to all 784 inputs.
- Uses **ReLU**: negative values become zero; positive values stay unchanged.
- Produces **128 values** that represent patterns learned from the input.

### 2. Second hidden layer

```python
model.add(Dense(128, activation='relu'))
```

Adds another layer of **128 neurons**.

It receives the 128 outputs from the previous layer and learns further combinations of those patterns. You don’t need to specify `input_shape` again because Keras infers it.

### 3. Dropout (original example; not used in these Dense-only experiments)

```python
model.add(Dropout(0.25))
```

During training, randomly sets **25% of the incoming values to zero** at each step.

This helps reduce **overfitting**, where the model memorizes training examples but performs poorly on new examples. Dropout is turned off when evaluating or making predictions.

### 4. Output layer

```python
model.add(Dense(10, activation='softmax'))
```

Creates **10 output neurons**, one for each class.

**Softmax** turns their results into probabilities that sum to 1. For example:

```text
Digit 0: 0.01
Digit 1: 0.02
Digit 2: 0.90
Other digits combined: 0.07
```

The model predicts **digit 2**, since it has the highest probability.

### 5. Configure training

```python
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
```

`compile()` configures how the model will learn; **it does not train it yet**.

| Setting | Meaning |
|---|---|
| `loss='sparse_categorical_crossentropy'` | Measures how far the predicted probabilities are from the correct class. Lower is better. |
| `optimizer='adam'` | Adjusts the weights and biases during training to reduce the loss. |
| `metrics=['accuracy']` | Reports the proportion of predictions that are correct. |

**Label format matters:** this notebook keeps integer labels, so it uses `sparse_categorical_crossentropy`. One-hot targets require `categorical_crossentropy` instead.

### 6. Display the model

```python
model.summary()
```

Prints the layers, their output shapes, and their parameter counts:

| Layer | Output shape | Trainable parameters |
|---|---|---:|
| First Dense | `(None, 128)` | 100,480 |
| Second Dense | `(None, 128)` | 16,512 |
| Dropout | `(None, 128)` | 0 |
| Output Dense | `(None, 10)` | 1,290 |
| **Total** | | **118,282** |

**`None` represents the batch size**, which can vary. Dense parameters are calculated as:

\[
(\text{number of inputs}\times\text{number of neurons})+\text{number of biases}
\]

Dropout has zero parameters because it has no weights or biases to learn.

The baseline flow here is **784 inputs → 128 neurons → 128 neurons → 10 class probabilities**. Trial B changes the first hidden layer to 256 neurons; its parameter count differs from the baseline table above. Training starts when you call `model.fit(...)`.
# Parameters are the weights and biases learned during training.
# Dense parameters = (inputs * neurons) + neurons (one bias per neuron).
# First Dense: 784 * 128 + 128 = 100,480
# Second Dense: 128 * 128 + 128 = 16,512
# Output Dense: 128 * 10 + 10 = 1,290
# Total: 118,282; Dropout, ReLU, and softmax add no trainable parameters.
#
# Overfitting: the model performs well on training data but poorly on unseen data.
# Common sign: training loss decreases while validation loss increases.
# Reduce it using more varied data, dropout, early stopping, fewer neurons/layers,
# weight regularization, and suitable data augmentation.

#Train

In [ ]:
trial_results = []
histories = {}

def run_mlp(config):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_mlp(config['hidden_units'], config['learning_rate'], config['l2'])
    checkpoint_path = CHECKPOINT_DIR / f"mlp_{config['name']}.keras"
    # Prevent a checkpoint from an earlier run being mistaken for this trial.
    if checkpoint_path.exists():
        checkpoint_path.unlink()
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor='val_loss', mode='min', save_best_only=True,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss', mode='min', patience=PATIENCE,
            restore_best_weights=True,
        ),
        keras.callbacks.TerminateOnNaN(),
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS, batch_size=config['batch_size'],
        callbacks=callbacks, verbose=2, shuffle=True,
    )
    train_seconds = time.perf_counter() - start
    values = history.history
    if not all(np.isfinite(values[key]).all() for key in ('loss', 'val_loss', 'accuracy', 'val_accuracy')):
        raise RuntimeError(f"Trial {config['name']} produced non-finite metrics; investigate before selecting a model.")
    if not checkpoint_path.exists():
        raise RuntimeError(f"No checkpoint saved for trial {config['name']}.")
    best_index = int(np.argmin(values['val_loss']))
    # Verify the metrics of the actual file that will be considered for deployment.
    saved_model = keras.models.load_model(checkpoint_path)
    val_loss, val_accuracy = saved_model.evaluate(X_val, y_val, verbose=0)
    row = {
        **config,
        'best_epoch': best_index + 1,
        'epochs_run': len(values['loss']),
        'val_loss': float(val_loss),
        'val_accuracy': float(val_accuracy),
        'train_accuracy_at_best_epoch': float(values['accuracy'][best_index]),
        'train_seconds': float(train_seconds),
        'parameters': int(saved_model.count_params()),
        'checkpoint': str(checkpoint_path.relative_to(PROJECT_ROOT)),
    }
    pd.DataFrame(values).rename_axis('epoch_index').to_csv(RESULT_DIR / f"history_{config['name']}.csv")
    histories[config['name']] = values
    print(f"Trial {config['name']}: best epoch {row['best_epoch']}, validation accuracy {val_accuracy:.4%}, training {train_seconds:.1f}s")
    return row

In [ ]:
for config in EXPERIMENTS:
    trial_results.append(run_mlp(config))
    pd.DataFrame(trial_results).to_csv(RESULT_DIR / 'tuning_results_partial.csv', index=False)

results = pd.DataFrame(trial_results)
results.to_csv(RESULT_DIR / 'tuning_results.csv', index=False)
print(results[['name', 'hidden_units', 'learning_rate', 'batch_size', 'l2',
               'best_epoch', 'epochs_run', 'val_accuracy', 'val_loss', 'train_seconds', 'parameters']].to_string(index=False))

# Compare validation results

In [ ]:
fig, axes = plt.subplots(len(EXPERIMENTS), 2, figsize=(12, 3 * len(EXPERIMENTS)))
for row, config in enumerate(EXPERIMENTS):
    h = histories[config['name']]
    epochs = np.arange(1, len(h['loss']) + 1)
    axes[row, 0].plot(epochs, h['loss'], label='Training')
    axes[row, 0].plot(epochs, h['val_loss'], label='Validation')
    axes[row, 0].set_title(f"Trial {config['name']} — loss")
    axes[row, 1].plot(epochs, h['accuracy'], label='Training')
    axes[row, 1].plot(epochs, h['val_accuracy'], label='Validation')
    axes[row, 1].set_title(f"Trial {config['name']} — accuracy")
    for ax in axes[row]:
        ax.set_xlabel('Epoch')
        ax.legend()
fig.tight_layout()
fig.savefig(RESULT_DIR / 'learning_curves.png', dpi=150)
plt.show()

# Save the selected model

In [ ]:
ranked = results.sort_values(
    ['val_accuracy', 'val_loss', 'parameters', 'name'],
    ascending=[False, True, True, True],
)
winner = ranked.iloc[0].to_dict()
FINAL_PATH = MODEL_DIR / 'mlp_final.keras'
# Copy the exact selected checkpoint; do not retrain on the test set.
shutil.copy2(PROJECT_ROOT / winner['checkpoint'], FINAL_PATH)
print('Chosen trial:', winner['name'])
print('Hidden units:', winner['hidden_units'])
print('Learning rate:', winner['learning_rate'], 'Batch size:', winner['batch_size'], 'L2:', winner['l2'])
print('Selected epoch:', winner['best_epoch'], 'Validation accuracy:', winner['val_accuracy'])
print('Saved model:', FINAL_PATH)

#Evaluate

In [ ]:
keras.backend.clear_session()
load_start = time.perf_counter()
mlp_loaded = keras.models.load_model(FINAL_PATH)
load_seconds = time.perf_counter() - load_start

# Warm up using validation data, leaving test evaluation to the next prediction.
_ = mlp_loaded.predict(X_val[:128], batch_size=128, verbose=0)
start = time.perf_counter()
test_probs = mlp_loaded.predict(X_test, batch_size=128, verbose=0)
inference_seconds = time.perf_counter() - start
assert test_probs.shape == (10000, 10)
assert np.isfinite(test_probs).all()
assert np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-5)
test_pred = np.argmax(test_probs, axis=1)
test_accuracy = float(accuracy_score(y_test, test_pred))
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, test_pred, average='macro', zero_division=0,
)
# Predictive cross-entropy excludes any L2 weight penalty.
true_class_probs = test_probs[np.arange(len(y_test)), y_test]
test_crossentropy = float(-np.log(np.clip(true_class_probs, 1e-7, 1.0)).mean())
report = classification_report(y_test, test_pred, labels=list(range(10)), output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, test_pred, labels=list(range(10)))

print(f'Test accuracy: {test_accuracy:.4%}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Predictive cross-entropy: {test_crossentropy:.4f}')
print(f'Model size: {FINAL_PATH.stat().st_size / (1024 ** 2):.2f} MiB')
print(f'Loading: {load_seconds:.2f}s; 10,000 predictions after warm-up: {inference_seconds:.2f}s')
print(classification_report(y_test, test_pred, labels=list(range(10)), digits=4, zero_division=0))
pd.DataFrame(report).T.to_csv(RESULT_DIR / 'classification_report.csv')
pd.DataFrame(cm, index=range(10), columns=range(10)).to_csv(RESULT_DIR / 'confusion_matrix.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Selected MLP — test confusion matrix')
fig.tight_layout()
fig.savefig(RESULT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

wrong = np.flatnonzero(test_pred != y_test)[:12]
if len(wrong):
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, wrong):
        ax.imshow(X_test_raw[index], cmap='gray')
        ax.set_title(f'True: {y_test[index]} / Pred: {test_pred[index]}')
    fig.tight_layout()
    fig.savefig(RESULT_DIR / 'misclassified_examples.png', dpi=150)
    plt.show()
else:
    print('No test errors in this run.')

# Verify reload and record results

In [ ]:
second_load = keras.models.load_model(FINAL_PATH, compile=False)
validation_probs_1 = mlp_loaded.predict(X_val[:32], verbose=0)
validation_probs_2 = second_load.predict(X_val[:32], verbose=0)
np.testing.assert_allclose(validation_probs_1, validation_probs_2, rtol=1e-5, atol=1e-6)
assert second_load.input_shape == (None, 784)
assert second_load.output_shape == (None, 10)
del second_load
print('Reload check passed; input (N, 784), output (N, 10).')

In [ ]:
metrics = {
    'model': 'MLP',
    'selected_trial': str(winner['name']),
    'best_epoch': int(winner['best_epoch']),
    'validation_accuracy': float(winner['val_accuracy']),
    'test_accuracy': test_accuracy,
    'test_predictive_crossentropy': test_crossentropy,
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'parameters': int(mlp_loaded.count_params()),
    'selected_trial_train_seconds': float(winner['train_seconds']),
    'total_search_train_seconds': float(results['train_seconds'].sum()),
    'load_seconds': float(load_seconds),
    'test_inference_seconds_10000_after_warmup': float(inference_seconds),
    'file_size_mib': FINAL_PATH.stat().st_size / (1024 ** 2),
}
metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'MNIST',
    'split': {'train': 50000, 'validation': 10000, 'test': 10000,
              'random_state': SEED, 'stratify': True},
    'preprocessing': {'input_shape': [784], 'dtype': 'float32',
                      'normalization': 'raw pixel / 255.0',
                      'polarity': 'bright digit on dark background',
                      'labels': 'integer digits 0-9'},
    'selection': 'Highest validation accuracy at each trial minimum-val-loss checkpoint; tie: lower val_loss, fewer parameters, trial name',
    'hyperparameters': next(c for c in EXPERIMENTS if c['name'] == winner['name']),
    'training': {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'seed': SEED},
    'environment': environment,
    'metrics': metrics,
}
(MODEL_DIR / 'mlp_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(RESULT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame([metrics]).to_csv(RESULT_DIR / 'comparison_row.csv', index=False)
(RESULT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))